# Geracao de `apartes_parlamentares/v1` no Google Colab

Gera a tabela relacional e o Parquet de apartes em plenario a partir dos raws em `raw/senado/plenario_apartes/metadata/` e `raw/camara/plenario_apartes/metadata/`. O resultado fica separado de `textos_parlamentares/v1` e nao cria corpus textual.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/falando_nela/data')
os.environ['FALANDO_NELA_DATA_ROOT'] = str(DATA_ROOT)

DATASET_VERSION = 'v1'
DATASET_ROOT = DATA_ROOT / 'processed' / 'apartes_parlamentares' / DATASET_VERSION
PARQUET_PATH = DATASET_ROOT / 'parquet' / 'apartes_parlamentares.parquet'
MANIFEST_ROOT = DATA_ROOT / 'processed' / 'manifests'
AUDIT_ROOT = DATA_ROOT / 'processed' / 'audits' / 'apartes_parlamentares'

RUN_ID = f"processed-apartes-parlamentares-v1-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
SOURCE = 'all'  # all, camara ou senado
RAW_RUN_IDS = []  # opcional: ['raw-run-id-1', 'raw-run-id-2']
OVERWRITE = True
LIMIT_RECORDS = None  # use um inteiro para validacao curta

print('DATA_ROOT:', DATA_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('PARQUET_PATH:', PARQUET_PATH)
print('RUN_ID:', RUN_ID)
print('SOURCE:', SOURCE)
print('RAW_RUN_IDS:', RAW_RUN_IDS or 'todos')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/pedblan/falando_nela.git'
REPO_DIR = Path('/content/falando_nela')
REPO_REF = ''  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags', '--prune'], check=True)
    if not REPO_REF:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

if REPO_REF:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Repo em:', Path.cwd())
subprocess.run(['git', 'status', '--short'], check=True)


## Conferir entradas

Confirma os raws de apartes e a dimensao `parlamentares/v1`. A dimensao de parlamentares melhora genero, partido e UF; se ela ainda nao existir, o processador continua e marca os matches possiveis sem inferir genero por nome.


In [ ]:
from pathlib import Path

raw_senado = sorted((DATA_ROOT / 'raw' / 'senado' / 'plenario_apartes' / 'metadata').glob('*.jsonl'))
raw_camara = sorted((DATA_ROOT / 'raw' / 'camara' / 'plenario_apartes' / 'metadata').glob('*.jsonl'))
periodos_jsonl = DATA_ROOT / 'processed' / 'parlamentares' / 'v1' / 'parlamentares_periodos.jsonl'
periodos_parquet = DATA_ROOT / 'processed' / 'parlamentares' / 'v1' / 'parquet' / 'parlamentares_periodos.parquet'

print('Raws Senado:', len(raw_senado))
for path in raw_senado[-5:]:
    print('  ', path)
print('Raws Camara:', len(raw_camara))
for path in raw_camara[-5:]:
    print('  ', path)
print('Parlamentares periodos JSONL:', periodos_jsonl.exists(), periodos_jsonl)
print('Parlamentares periodos Parquet:', periodos_parquet.exists(), periodos_parquet)

if not raw_senado and SOURCE in {'all', 'senado'}:
    raise FileNotFoundError('Nenhum raw do Senado encontrado em metadata/. Rode o caderno de coleta do Senado primeiro.')
if not raw_camara and SOURCE in {'all', 'camara'}:
    raise FileNotFoundError('Nenhum raw da Camara encontrado em metadata/. Rode o caderno de coleta da Camara primeiro.')
if not periodos_jsonl.exists() and not periodos_parquet.exists():
    print('Aviso: parlamentares/v1 ausente. Genero ficara vazio e matches por nome serao limitados.')


## Gerar tabela e Parquet

O processador ignora probes anuais/trimestrais para nao duplicar relacoes, deduplica `aparte_id` e grava JSONL, Parquet, manifesto e auditorias anuais.


In [ ]:
import json
from processamento.apartes_parlamentares import process_apartes_data_root

manifest = process_apartes_data_root(
    DATA_ROOT,
    run_id=RUN_ID,
    raw_run_ids=RAW_RUN_IDS or None,
    source=SOURCE,
    overwrite=OVERWRITE,
    limit_records=LIMIT_RECORDS,
)

print(json.dumps({
    'manifest_path': manifest['manifest_path'],
    'output_records': manifest['output_records'],
    'input_record_counts': manifest['input_record_counts'],
    'output_record_counts': manifest['output_record_counts'],
    'skipped_counts': manifest['skipped_counts'],
    'parquet_files': manifest['parquet_files'],
    'audit_root': manifest['audit_root'],
    'parlamentares_index': manifest['parlamentares_index'],
}, ensure_ascii=False, indent=2))


## Validar saidas

Checa unicidade, schema minimo, ausencia de texto de corpus e coerencia de datas. Depois mostra uma primeira tabela anual para a analise.


In [ ]:
import pandas as pd

jsonl_path = DATASET_ROOT / 'apartes_parlamentares.jsonl'
parquet_path = DATASET_ROOT / 'parquet' / 'apartes_parlamentares.parquet'
manifest_path = MANIFEST_ROOT / f'{RUN_ID}-apartes-parlamentares.json'
audit_path = AUDIT_ROOT / RUN_ID

for required in [jsonl_path, parquet_path, manifest_path, audit_path / 'contagens_anuais.csv', audit_path / 'match_status.csv', audit_path / 'cobertura_parlamentares.csv']:
    if not required.exists():
        raise FileNotFoundError(required)

expected_columns = [
    'aparte_id', 'dataset_version', 'source', 'casa', 'data', 'ano', 'mes',
    'pronunciamento_id', 'discurso_chave', 'sessao_id', 'tipo_sessao', 'fase_sessao',
    'orador_id', 'orador_nome', 'orador_genero', 'orador_partido', 'orador_uf',
    'aparteante_id', 'aparteante_nome', 'aparteante_genero', 'aparteante_partido', 'aparteante_uf',
    'url_texto', 'url_diario', 'url_origem', 'match_status',
    'raw_run_id', 'raw_record_type', 'raw_source_id', 'raw_partition', 'raw_collected_at',
    'raw_checksum', 'raw_path', 'raw_response_url',
]

df = pd.read_parquet(parquet_path)
missing = [column for column in expected_columns if column not in df.columns]
if missing:
    raise AssertionError(f'Colunas ausentes: {missing}')
if 'texto' in df.columns:
    raise AssertionError('O dataset de apartes nao deve conter campo texto.')
if not df['aparte_id'].is_unique:
    duplicated = df.loc[df['aparte_id'].duplicated(), 'aparte_id'].head().tolist()
    raise AssertionError(f'aparte_id duplicado: {duplicated}')
if not set(df['source'].dropna().unique()).issubset({'camara', 'senado'}):
    raise AssertionError(f'Sources inesperadas: {df["source"].dropna().unique()}')

with_dates = df.dropna(subset=['data']).copy()
if not with_dates.empty:
    parsed_dates = pd.to_datetime(with_dates['data'], errors='raise')
    if not (parsed_dates.dt.year.astype('Int64').astype(str).to_numpy() == with_dates['ano'].astype(str).to_numpy()).all():
        raise AssertionError('Ano incoerente com data em pelo menos uma linha.')
    if not (parsed_dates.dt.month.astype('Int64').astype(str).to_numpy() == with_dates['mes'].astype(str).to_numpy()).all():
        raise AssertionError('Mes incoerente com data em pelo menos uma linha.')

print('Linhas:', len(df))
print('Parquet:', parquet_path)
print('Manifest:', manifest_path)
print('Auditoria:', audit_path)
display(df.head(20))


In [ ]:
contagens = pd.read_csv(AUDIT_ROOT / RUN_ID / 'contagens_anuais.csv')
status = pd.read_csv(AUDIT_ROOT / RUN_ID / 'match_status.csv')
cobertura = pd.read_csv(AUDIT_ROOT / RUN_ID / 'cobertura_parlamentares.csv')

print('Contagens anuais')
display(contagens.sort_values(['source', 'ano', 'aparteante_genero', 'aparteante_partido', 'aparteante_uf']).head(50))
print('Match status')
display(status.sort_values(['source', 'ano', 'match_status']).head(50))
print('Cobertura parlamentares')
display(cobertura.sort_values(['source', 'ano', 'papel', 'tem_genero']).head(50))
